
# Experiment 17 — Long-Horizon Official PatchTST + Frozen Historical Memory

## 연구 질문

이번 실험은 Experiment 16에서 사용한 방법을 전혀 수정하지 않고 예측 horizon만 확장합니다.

$$
H \in \{192, 336, 720\}
$$

핵심 질문은 다음과 같습니다.

$$
\boxed{
\text{Does frozen historical memory become more complementary as the forecasting horizon grows?}
}
$$

대상 데이터셋은 다음 두 개입니다.

- ETTm1
- ETTh1

---

## 고정하는 요소

### Direct forecaster

각 horizon에서 공식 PatchTST supervised recipe를 그대로 사용합니다.

$$
L_D = 336
$$

ETTh1과 ETTm1의 공식 shell script는 모두
\(H=96,192,336,720\)에 대해 동일한 dataset-specific architecture를 사용합니다.

### Retrieval module

Experiment 13에서 학습된 horizon별 EmbeddingOnly retriever를 그대로 사용합니다.

$$
L_R = 96
$$

- patch length 16
- stride 16
- embedding dimension 64
- Top-10 retrieval
- memory stride 24
- uniform retrieved-future aggregation

### Adaptive integration

Experiment 16과 동일합니다.

$$
\hat{\mathbf y}
=
(1-\alpha_q)\hat{\mathbf y}_{\mathrm{PatchTST}}
+
\alpha_q\hat{\mathbf y}_{\mathrm{retrieval}}
$$

Cross-fitted adaptive gate를 사용하고, validation에서 scalar prior와 shrinkage를 선택합니다.

$$
\alpha_{\mathrm{final}}
=
(1-\lambda)\alpha_0
+
\lambda \alpha_{\mathrm{gate}}
$$

---

## Cross-fitting

훈련 구간 내부에서 세 개의 chronological fold를 사용합니다.

$$
0.55 \rightarrow 0.70,\qquad
0.70 \rightarrow 0.85,\qquad
0.85 \rightarrow 1.00
$$

각 fold의 official PatchTST는 full official model의 validation-best epoch 수만큼 고정 학습합니다.

따라서 OOF target을 이용해 direct model의 epoch를 선택하지 않습니다.

---

## Test protocol

- all valid test origins
- stride 1
- all 7 channels
- direct PatchTST는 standard train split으로만 학습
- retrieval test memory는 train + validation history까지만 사용
- test label은 retrieval memory에 절대 포함하지 않음

---

## 이번 실험에서 튜닝하지 않는 것

다음 항목은 모두 Experiment 16과 동일하게 고정합니다.

- retriever architecture
- retriever objective
- retrieval query length
- Top-K
- memory stride
- gate feature set
- gate architecture
- scalar alpha grid
- shrinkage lambda grid
- cross-fit folds

즉 이번 실험은 architecture search가 아니라 **horizon generalization test**입니다.


In [1]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext
import gc
import importlib
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 420)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASETS = ["ETTm1", "ETTh1"]
HORIZONS = [192, 336, 720]

DIRECT_SEQ_LEN = 336
RET_SEQ_LEN = 96
LABEL_LEN = 48

OFFICIAL_SEED = 2021
CROSSFIT_SEED = 1313

# Frozen retrieval settings
TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4
FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

# Frozen representation
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DIM = 64
REP_DROPOUT = 0.1
REP_NUM_PATCHES = 1 + (RET_SEQ_LEN - REP_PATCH_LEN) // REP_PATCH_STRIDE

# Frozen gate
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(np.arange(0.0, 1.0001, 0.1), 10)
LAMBDA_GRID = np.array([0.0, 0.25, 0.50, 0.75, 1.00], dtype=np.float32)

FULL_ANCHOR_BATCH = 8
EPS = 1e-8
RETRIEVER_USE_AMP = torch.cuda.is_available()

FROZEN_RET_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening"
)

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "official_patchtst_plus_frozen_retrieval_long_horizon"
)

DIRS = {
    "full_direct": ROOT / "full_direct",
    "fold_direct": ROOT / "fold_direct",
    "oof": ROOT / "oof",
    "gate": ROOT / "gate",
    "history": ROOT / "history",
    "paired": ROOT / "paired_test",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

RESUME = True
FORCE = False

print("Device:", DEVICE)
print("Datasets:", DATASETS)
print("Horizons:", HORIZONS)
print("Frozen retriever root:", FROZEN_RET_ROOT)
print("Output:", ROOT)


Device: cuda
Datasets: ['ETTm1', 'ETTh1']
Horizons: [192, 336, 720]
Frozen retriever root: /data/dataset/strong_forecaster/multidataset_crossfit_screening
Output: /data/dataset/strong_forecaster/official_patchtst_plus_frozen_retrieval_long_horizon


## 1. Official PatchTST implementation

In [2]:

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p for p in OFFICIAL_REPO_CANDIDATES
        if (
            p / "PatchTST_supervised" / "models" / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    raise FileNotFoundError(
        "Official PatchTST repository was not found. "
        "Experiment 15 should have cloned it already."
    )

SUPERVISED_ROOT = OFFICIAL_REPO / "PatchTST_supervised"

for module_name in list(sys.modules.keys()):
    if (
        module_name == "models"
        or module_name.startswith("models.")
        or module_name == "layers"
        or module_name.startswith("layers.")
    ):
        del sys.modules[module_name]

if str(SUPERVISED_ROOT) in sys.path:
    sys.path.remove(str(SUPERVISED_ROOT))

sys.path.insert(0, str(SUPERVISED_ROOT))

patchtst_module = importlib.import_module("models.PatchTST")
OfficialPatchTST = patchtst_module.Model

actual_model_file = Path(patchtst_module.__file__).resolve()
expected_model_file = (SUPERVISED_ROOT / "models" / "PatchTST.py").resolve()

print("Imported:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong PatchTST implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print("PASS: official PatchTST implementation is active.")


Imported: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py
PASS: official PatchTST implementation is active.


## 2. Standard ETT data

In [3]:

DATA_PATHS = {
    "ETTh1": next(
        (
            p for p in [
                Path(
                    "/data/Time-Series-Library/"
                    "dataset/ETT-small/ETTh1.csv"
                ),
                Path("/data/dataset/ETTh1.csv"),
            ]
            if p.is_file()
        ),
        None,
    ),
    "ETTm1": Path("/data/dataset/ETTm1.csv"),
}


def ett_unit(name):
    if name == "ETTh1":
        return 30 * 24
    if name == "ETTm1":
        return 30 * 24 * 4
    raise ValueError(name)


def split_bounds(name):
    u = ett_unit(name)
    return 12 * u, 16 * u, 20 * u


def load_data(name):
    path = DATA_PATHS[name]

    if path is None or not path.is_file():
        raise FileNotFoundError(f"{name} dataset not found: {path}")

    df = pd.read_csv(path)

    if "date" not in df.columns:
        raise ValueError(
            f"{name}: expected standard ETT CSV with a date column."
        )

    cols = [c for c in df.columns if c != "date"]
    raw_all = df[cols].to_numpy(dtype=np.float32)

    tr, va, te = split_bounds(name)

    if len(raw_all) < te:
        raise ValueError(
            f"{name}: standard split needs {te} rows, "
            f"file has {len(raw_all)}."
        )

    raw = raw_all[:te].copy()

    if raw.shape[1] != 7:
        raise ValueError(
            f"{name}: expected 7 channels, found {raw.shape[1]}."
        )

    mu = raw[:tr].mean(axis=0).astype(np.float32)
    sd = raw[:tr].std(axis=0).astype(np.float32)

    if np.any(sd <= 1e-6):
        raise ValueError(f"{name}: degenerate training channel.")

    z = ((raw - mu[None, :]) / sd[None, :]).astype(np.float32)

    return {
        "name": name,
        "path": path,
        "columns": cols,
        "raw": raw,
        "z": z,
        "mean": mu,
        "std": sd,
        "n_channels": raw.shape[1],
        "train_end": tr,
        "val_end": va,
        "test_end": te,
    }


DATA = {name: load_data(name) for name in DATASETS}

display(pd.DataFrame([
    {
        "Dataset": name,
        "Path": str(d["path"]),
        "Channels": d["n_channels"],
        "TrainEnd": d["train_end"],
        "ValEnd": d["val_end"],
        "TestEnd": d["test_end"],
    }
    for name, d in DATA.items()
]))


,Dataset,Path,Channels,TrainEnd,ValEnd,TestEnd
0,ETTm1,/data/dataset/ETTm1.csv,7,34560,46080,57600
1,ETTh1,/data/Time-Series-Library/dataset/ETT-small/ET...,7,8640,11520,14400


## 3. Official PatchTST recipes

In [4]:

OFFICIAL_RECIPES = {
    "ETTh1": {
        "enc_in": 7,
        "e_layers": 3,
        "n_heads": 4,
        "d_model": 16,
        "d_ff": 128,
        "dropout": 0.3,
        "fc_dropout": 0.3,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 100,
        "learning_rate": 1e-4,
        "lradj": "type3",
        "pct_start": 0.3,
    },
    "ETTm1": {
        "enc_in": 7,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 20,
        "learning_rate": 1e-4,
        "lradj": "TST",
        "pct_start": 0.4,
    },
}


def official_config(name, h):
    r = OFFICIAL_RECIPES[name]

    return SimpleNamespace(
        enc_in=r["enc_in"],
        seq_len=DIRECT_SEQ_LEN,
        pred_len=h,
        e_layers=r["e_layers"],
        n_heads=r["n_heads"],
        d_model=r["d_model"],
        d_ff=r["d_ff"],
        dropout=r["dropout"],
        fc_dropout=r["fc_dropout"],
        head_dropout=r["head_dropout"],
        individual=0,
        patch_len=r["patch_len"],
        stride=r["stride"],
        padding_patch="end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_official_model(name, h):
    return OfficialPatchTST(
        official_config(name, h)
    ).float().to(DEVICE)


display(pd.DataFrame(OFFICIAL_RECIPES).T)


,enc_in,e_layers,n_heads,d_model,d_ff,dropout,fc_dropout,head_dropout,patch_len,stride,batch_size,train_epochs,patience,learning_rate,lradj,pct_start
ETTh1,7,3,4,16,128,0.3,0.3,0.0,16,8,128,100,100,0.0001,type3,0.3
ETTm1,7,3,16,128,256,0.2,0.2,0.0,16,8,128,100,20,0.0001,TST,0.4


## 4. Official ETT dataset and full direct training

In [5]:

class ETTForecastDataset(Dataset):
    def __init__(self, name, flag, h):
        super().__init__()

        if flag not in {"train", "val", "test"}:
            raise ValueError(flag)

        self.name = name
        self.flag = flag
        self.h = int(h)

        d = DATA[name]
        raw = d["raw"]

        scaler = StandardScaler()
        scaler.fit(raw[:d["train_end"]])

        scaled = scaler.transform(raw).astype(np.float32)

        border1s = [
            0,
            d["train_end"] - DIRECT_SEQ_LEN,
            d["val_end"] - DIRECT_SEQ_LEN,
        ]

        border2s = [
            d["train_end"],
            d["val_end"],
            d["test_end"],
        ]

        idx = {"train": 0, "val": 1, "test": 2}[flag]

        self.data_x = scaled[border1s[idx]:border2s[idx]]

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + DIRECT_SEQ_LEN

        r_begin = s_end - LABEL_LEN
        r_end = r_begin + LABEL_LEN + self.h

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_x[r_begin:r_end]

        return (
            torch.from_numpy(seq_x),
            torch.from_numpy(seq_y),
        )

    def __len__(self):
        return (
            len(self.data_x)
            - DIRECT_SEQ_LEN
            - self.h
            + 1
        )


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False


def make_official_loaders(name, h):
    r = OFFICIAL_RECIPES[name]

    train_ds = ETTForecastDataset(name, "train", h)
    val_ds = ETTForecastDataset(name, "val", h)
    test_ds = ETTForecastDataset(name, "test", h)

    return {
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train": DataLoader(
            train_ds,
            batch_size=r["batch_size"],
            shuffle=True,
            num_workers=0,
            drop_last=True,
        ),
        "val": DataLoader(
            val_ds,
            batch_size=r["batch_size"],
            shuffle=True,
            num_workers=0,
            drop_last=True,
        ),
        "full_test": DataLoader(
            test_ds,
            batch_size=r["batch_size"],
            shuffle=False,
            num_workers=0,
            drop_last=False,
        ),
    }


@torch.no_grad()
def evaluate_direct_loader(model, loader, h):
    model.eval()

    batch_mse = []
    sse = 0.0
    sae = 0.0
    count = 0

    for batch_x, batch_y in loader:
        x = batch_x.float().to(DEVICE)
        y = batch_y[:, -h:, :].float().to(DEVICE)

        pred = model(x)[:, -h:, :]
        err = pred - y

        batch_mse.append(float((err ** 2).mean().item()))
        sse += float((err ** 2).sum().item())
        sae += float(err.abs().sum().item())
        count += err.numel()

        del x, y, pred, err

    return {
        "BatchAverageMSE": float(np.mean(batch_mse)),
        "MSE": sse / count,
        "MAE": sae / count,
        "Batches": len(batch_mse),
    }


def adjust_type3_lr(optimizer, base_lr, epoch):
    lr = (
        base_lr
        if epoch < 3
        else base_lr * (0.9 ** (epoch - 3))
    )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


def full_direct_ckpt_path(name, h):
    return (
        DIRS["full_direct"]
        / f"{name}_L336_H{h}_official_recipe_seed2021.pt"
    )


def train_or_load_full_direct(name, h):
    path = full_direct_ckpt_path(name, h)

    if path.exists() and RESUME and not FORCE:
        ckpt = torch.load(path, map_location=DEVICE)

        model = build_official_model(name, h)
        model.load_state_dict(ckpt["StateDict"])
        model.eval()

        print(
            f"Loaded full official direct: {name} H={h}, "
            f"best={ckpt['BestValMSE']:.6f}@{ckpt['BestEpoch']}"
        )

        return model, ckpt

    r = OFFICIAL_RECIPES[name]

    set_seed(OFFICIAL_SEED)

    loaders = make_official_loaders(name, h)
    model = build_official_model(name, h)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
    )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        steps_per_epoch=len(loaders["train"]),
        pct_start=r["pct_start"],
        epochs=r["train_epochs"],
        max_lr=r["learning_rate"],
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, r["train_epochs"] + 1):
        model.train()
        losses = []
        t0 = time.time()

        for batch_x, batch_y in loaders["train"]:
            x = batch_x.float().to(DEVICE)
            y = batch_y[:, -h:, :].float().to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            pred = model(x)[:, -h:, :]
            loss = F.mse_loss(pred, y)

            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

            if r["lradj"] == "TST":
                current = onecycle.get_last_lr()[0]
                for group in optimizer.param_groups:
                    group["lr"] = current
                onecycle.step()

            del x, y, pred, loss

        val = evaluate_direct_loader(
            model,
            loaders["val"],
            h,
        )

        val_for_selection = val["BatchAverageMSE"]

        if val_for_selection < best_val - 1e-12:
            best_val = val_for_selection
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if r["lradj"] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r["learning_rate"],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[0]

        history.append({
            "Epoch": epoch,
            "TrainMSE": float(np.mean(losses)),
            "ValMSE": val_for_selection,
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": lr,
            "Seconds": time.time() - t0,
        })

        print(
            f"Full {name:6s} H={h:3d} "
            f"ep={epoch:03d}/{r['train_epochs']} "
            f"train={np.mean(losses):.6f} "
            f"val={val_for_selection:.6f} "
            f"best={best_val:.6f}@{best_epoch} "
            f"wait={wait}/{r['patience']}"
        )

        pd.DataFrame(history).to_csv(
            DIRS["history"] / f"{name}_H{h}_full_direct_history.csv",
            index=False,
        )

        if wait >= r["patience"]:
            print("Early stopping.")
            break

    if best_state is None:
        raise RuntimeError(f"{name} H={h}: no best state.")

    model.load_state_dict(best_state)
    model.eval()

    ckpt = {
        "Dataset": name,
        "Horizon": h,
        "BestEpoch": best_epoch,
        "BestValMSE": best_val,
        "StateDict": best_state,
    }

    torch.save(ckpt, path)

    del loaders

    return model, ckpt



## 5. Frozen predictive retriever

Representation architecture와 retrieval rule은 Experiment 16과 동일합니다.

$$
s(q,i)
=
\gamma\,
\cos\!\left(
z_q,z_i
\right)
$$

Top-10 candidate의 future residual을 균등 평균하여 historical forecast를 만듭니다.


In [6]:

def inv_softplus(x):
    return math.log(math.exp(float(x)) - 1.0)


class PredictivePatchEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=REP_D_MODEL,
            nhead=REP_N_HEADS,
            dim_feedforward=REP_D_FF,
            dropout=REP_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=REP_LAYERS,
        )

        self.norm = nn.LayerNorm(REP_D_MODEL)
        self.proj = nn.Linear(REP_D_MODEL, REP_DIM)

    def forward(self, x):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(p)
            + self.pos_embed[:, :p.shape[1]]
        )

        h = self.encoder(h).mean(dim=1)
        h = self.proj(self.norm(h))

        return F.normalize(h, dim=-1, eps=1e-8)


class EmbeddingOnlyRetriever(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = PredictivePatchEncoder()

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(1.0),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(self):
        return F.softplus(self.raw_gamma)

    def encode(self, x):
        return self.encoder(x)


class CrossFitAdaptiveGate(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(GATE_DIM, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

        nn.init.normal_(
            self.net[-1].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[-1].bias,
            math.log(0.1 / 0.9),
        )

    def forward(self, x):
        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


def load_torch(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def full_retriever_ckpt_path(name, h):
    return (
        FROZEN_RET_ROOT
        / "full_retriever"
        / f"{name}_H{h}_seed0.pt"
    )


def fold_retriever_ckpt_path(name, h, fold):
    return (
        FROZEN_RET_ROOT
        / "fold_retriever"
        / f"{name}_H{h}_F{fold}_seed0.pt"
    )


def load_frozen_retriever(path):
    ckpt = load_torch(path)

    model = EmbeddingOnlyRetriever().to(DEVICE)
    model.load_state_dict(ckpt["StateDict"])
    model.eval()

    return model, ckpt


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


## 6. Preflight frozen retriever checkpoints

In [7]:

preflight_rows = []

for name in DATASETS:
    for h in HORIZONS:
        preflight_rows.append({
            "Dataset": name,
            "Horizon": h,
            "Kind": "Full retriever",
            "Path": full_retriever_ckpt_path(name, h),
        })

        for fold in range(1, 4):
            preflight_rows.append({
                "Dataset": name,
                "Horizon": h,
                "Kind": f"Fold retriever F{fold}",
                "Path": fold_retriever_ckpt_path(name, h, fold),
            })

preflight = pd.DataFrame(preflight_rows)

preflight["Exists"] = preflight["Path"].map(
    lambda p: Path(p).is_file()
)

display(preflight)

missing = preflight[~preflight["Exists"]]

if len(missing):
    print("\nMissing frozen checkpoints:")
    for p in missing["Path"]:
        print(" -", p)

    raise FileNotFoundError(
        "Some frozen retriever checkpoints are missing. "
        "Do not retrain selectively after seeing test results; "
        "restore the corresponding Experiment 13 checkpoints first."
    )

print("PASS: all frozen horizon-specific retriever checkpoints are available.")


,Dataset,Horizon,Kind,Path,Exists
0,ETTm1,192,Full retriever,/data/dataset/strong_forecaster/multidataset_c...,True
1,ETTm1,192,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
2,ETTm1,192,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True
3,ETTm1,192,Fold retriever F3,/data/dataset/strong_forecaster/multidataset_c...,True
4,ETTm1,336,Full retriever,/data/dataset/strong_forecaster/multidataset_c...,True
5,ETTm1,336,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
6,ETTm1,336,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True
7,ETTm1,336,Fold retriever F3,/data/dataset/strong_forecaster/multidataset_c...,True
8,ETTm1,720,Full retriever,/data/dataset/strong_forecaster/multidataset_c...,True
9,ETTm1,720,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True


PASS: all frozen horizon-specific retriever checkpoints are available.


## 7. Retrieval memory and query features

In [8]:

def eval_anchors(start, end, h, stride=1):
    return np.arange(
        max(int(start), DIRECT_SEQ_LEN),
        int(end) - int(h) + 1,
        int(stride),
        dtype=np.int64,
    )


def batch_pattern(x):
    x = np.asarray(x, np.float32)

    xc = x - x.mean(axis=-1, keepdims=True)
    n = np.linalg.norm(xc, axis=-1, keepdims=True)

    return np.where(
        n > EPS,
        xc / np.maximum(n, EPS),
        0.0,
    ).astype(np.float32)


def context7(x):
    x = np.asarray(x, np.float32)

    short = max(8, RET_SEQ_LEN // 4)

    m = x.mean(axis=-1)
    s = x.std(axis=-1) + EPS

    f1 = (x[..., -1] - m) / s
    f2 = (x[..., -short:].mean(axis=-1) - m) / s
    f3 = (x[..., -1] - x[..., -short]) / s
    f4 = (x[..., -1] - x[..., 0]) / s

    df = np.diff(x, axis=-1)
    ds = np.diff(x[..., -short:], axis=-1)

    f5 = (
        (ds.std(axis=-1) + EPS)
        / (df.std(axis=-1) + EPS)
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )
    t = t - t.mean()

    f6 = (
        np.sum(
            t * (x - m[..., None]),
            axis=-1,
        )
        / (np.sum(t * t) + EPS)
    ) / s

    a = x[..., :-1]
    b = x[..., 1:]

    a = a - a.mean(axis=-1, keepdims=True)
    b = b - b.mean(axis=-1, keepdims=True)

    f7 = (
        np.sum(a * b, axis=-1)
        / (
            np.sqrt(
                np.sum(a * a, axis=-1)
                * np.sum(b * b, axis=-1)
            )
            + EPS
        )
    )

    return np.stack(
        [f1, f2, f3, f4, f5, f6, f7],
        axis=-1,
    ).astype(np.float32)


def prefix_norm(raw, prefix):
    mu = raw[:prefix].mean(axis=0).astype(np.float32)
    sd = raw[:prefix].std(axis=0).astype(np.float32)

    if np.any(sd <= 1e-6):
        raise ValueError("Degenerate prefix channel.")

    return (
        (raw - mu[None, :])
        / sd[None, :]
    ).astype(np.float32)


def extract_channel(z, c, aa, h):
    aa = np.asarray(aa, np.int64)

    pi = (
        aa[:, None]
        - RET_SEQ_LEN
        + np.arange(RET_SEQ_LEN)[None, :]
    )

    fi = (
        aa[:, None]
        + np.arange(h)[None, :]
    )

    past = z[pi, c].astype(np.float32)
    future = z[fi, c].astype(np.float32)

    current = z[aa - 1, c].astype(np.float32)

    future_residual = (
        future
        - current[:, None]
    ).astype(np.float32)

    return past, future_residual


def build_memory(z, channels, boundary, h):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(boundary) - h + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(memory_anchors) < TOP_K:
        raise ValueError("Insufficient retrieval memory.")

    past = np.empty(
        (channels, len(memory_anchors), RET_SEQ_LEN),
        dtype=np.float32,
    )

    pattern = np.empty_like(past)

    future = np.empty(
        (channels, len(memory_anchors), h),
        dtype=np.float32,
    )

    for c in range(channels):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            h,
        )

        past[c] = p
        pattern[c] = batch_pattern(p)
        future[c] = f

    return {
        "anchors": memory_anchors,
        "past": past,
        "pattern": pattern,
        "future": future,
        "M": len(memory_anchors),
        "boundary": int(boundary),
    }


@torch.no_grad()
def encode_np(model, x, chunk=512):
    parts = []

    for i in range(0, len(x), chunk):
        t = torch.from_numpy(
            x[i:i+chunk]
        ).to(DEVICE)

        with ret_amp():
            e = model.encode(t).float()

        parts.append(e)

        del t, e

    return torch.cat(parts, dim=0)


@torch.no_grad()
def memory_gpu(model, memory, channels):
    emb = torch.empty(
        (
            channels,
            memory["M"],
            REP_DIM,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    for c in range(channels):
        emb[c] = encode_np(
            model,
            memory["past"][c],
        )

    return {
        "emb": emb,
        "pattern": torch.from_numpy(
            memory["pattern"]
        ).to(DEVICE),
        "future": torch.from_numpy(
            memory["future"]
        ).to(DEVICE),
    }


def query_pairs(z, aa, cc, h):
    aa = np.asarray(aa, np.int64)
    cc = np.asarray(cc, np.int64)

    pi = (
        aa[:, None]
        - RET_SEQ_LEN
        + np.arange(RET_SEQ_LEN)[None, :]
    )

    fi = (
        aa[:, None]
        + np.arange(h)[None, :]
    )

    past = z[
        pi,
        cc[:, None],
    ].astype(np.float32)

    future = z[
        fi,
        cc[:, None],
    ].astype(np.float32)

    current = z[
        aa - 1,
        cc,
    ].astype(np.float32)

    true_residual = (
        future
        - current[:, None]
    ).astype(np.float32)

    return (
        past,
        batch_pattern(past),
        context7(past),
        true_residual,
    )


@torch.no_grad()
def retrieve(model, memory_gpu_obj, z, aa, cc, h):
    past, pattern, ctx, true = query_pairs(
        z,
        aa,
        cc,
        h,
    )

    past_t = torch.from_numpy(past).to(DEVICE)
    pattern_t = torch.from_numpy(pattern).to(DEVICE)

    with ret_amp():
        qemb = model.encode(past_t)

    qemb = qemb.float()

    memb = memory_gpu_obj["emb"][cc]

    sim = torch.bmm(
        qemb[:, None, :],
        memb.transpose(1, 2),
    ).squeeze(1)

    score = model.gamma * sim

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(cc),
        device=DEVICE,
    )[:, None]

    pfull = torch.bmm(
        pattern_t[:, None, :],
        memory_gpu_obj["pattern"][cc].transpose(1, 2),
    ).squeeze(1)

    return {
        "score": score[row, idx],
        "sim": sim[row, idx],
        "pattern": pfull[row, idx],
        "cand": memory_gpu_obj["future"][
            cc[:, None],
            idx,
        ],
        "ctx": torch.from_numpy(ctx).to(DEVICE),
        "true": torch.from_numpy(true).to(DEVICE),
    }



## 8. Strong PatchTST prediction as residual

Retrieval forecast는 query의 마지막 관측값을 기준으로 한 residual trajectory입니다.

공정한 결합을 위해 PatchTST 출력도 같은 residual 좌표로 변환합니다.

$$
\Delta\hat{\mathbf y}_{\mathrm{direct}}
=
\hat{\mathbf y}_{\mathrm{direct}}
-
x_{t}
$$


In [9]:

@torch.no_grad()
def direct_residual(model, z, aa, h):
    aa = np.asarray(aa, np.int64)

    pi = (
        aa[:, None]
        - DIRECT_SEQ_LEN
        + np.arange(DIRECT_SEQ_LEN)[None, :]
    )

    x = torch.from_numpy(
        z[pi, :].astype(np.float32)
    ).to(DEVICE)

    pred = model(x).float()

    residual = (
        pred
        - x[:, -1:, :].float()
    )

    del x, pred

    return residual


@torch.no_grad()
def direct_full_test_metric(model, data, h):
    aa = eval_anchors(
        data["val_end"],
        data["test_end"],
        h,
        stride=1,
    )

    sse = 0.0
    sae = 0.0
    n = 0

    C = data["n_channels"]

    for i in range(
        0,
        len(aa),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[i:i+FULL_ANCHOR_BATCH]

        d3 = direct_residual(
            model,
            data["z"],
            a,
            h,
        )

        A = len(a)

        pa = np.repeat(a, C)
        pc = np.tile(
            np.arange(C, dtype=np.int64),
            A,
        )

        _, _, _, true = query_pairs(
            data["z"],
            pa,
            pc,
            h,
        )

        d = d3.permute(
            0,
            2,
            1,
        ).reshape(-1, h)

        y = torch.from_numpy(true).to(DEVICE)
        e = d - y

        sse += float((e * e).sum())
        sae += float(e.abs().sum())
        n += e.numel()

        del d3, d, y, e

    return sse / n, sae / n, len(aa)


## 9. Cross-fit official PatchTST

In [10]:

def fold_direct_path(name, h, fold):
    return (
        DIRS["fold_direct"]
        / f"{name}_H{h}_F{fold}_official_direct.pt"
    )


def direct_train_anchors(end, h):
    return np.arange(
        DIRECT_SEQ_LEN,
        int(end) - h + 1,
        dtype=np.int64,
    )


def make_direct_batch(z, aa, h):
    aa = np.asarray(aa, np.int64)

    pi = (
        aa[:, None]
        - DIRECT_SEQ_LEN
        + np.arange(DIRECT_SEQ_LEN)[None, :]
    )

    fi = (
        aa[:, None]
        + np.arange(h)[None, :]
    )

    return (
        z[pi, :].astype(np.float32),
        z[fi, :].astype(np.float32),
    )


def train_fold_official_direct(
    name,
    h,
    z,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(name, h, fold)

    if path.exists() and RESUME and not FORCE:
        ckpt = load_torch(path)

        model = build_official_model(name, h)
        model.load_state_dict(ckpt["StateDict"])
        model.eval()

        print("Loaded fold direct:", path.name)

        return model, ckpt

    r = OFFICIAL_RECIPES[name]

    seed = (
        CROSSFIT_SEED
        + h * 100
        + int(prefix) % 997
    )

    set_seed(seed)

    model = build_official_model(name, h)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
    )

    train_aa = direct_train_anchors(prefix, h)

    steps_per_epoch = (
        len(train_aa)
        // r["batch_size"]
    )

    if steps_per_epoch < 1:
        raise ValueError(
            f"{name} H={h} F{fold}: insufficient training windows."
        )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        steps_per_epoch=steps_per_epoch,
        pct_start=r["pct_start"],
        epochs=r["train_epochs"],
        max_lr=r["learning_rate"],
    )

    rng = np.random.default_rng(seed + 1)
    history = []

    for epoch in range(1, int(fixed_epochs) + 1):
        model.train()

        order = rng.permutation(len(train_aa))

        usable = (
            len(order)
            // r["batch_size"]
        ) * r["batch_size"]

        order = order[:usable]
        losses = []

        for left in range(
            0,
            usable,
            r["batch_size"],
        ):
            ids = order[
                left:left+r["batch_size"]
            ]

            x_np, y_np = make_direct_batch(
                z,
                train_aa[ids],
                h,
            )

            x = torch.from_numpy(x_np).to(DEVICE)
            y = torch.from_numpy(y_np).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            pred = model(x)
            loss = F.mse_loss(pred, y)

            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

            if r["lradj"] == "TST":
                current = onecycle.get_last_lr()[0]

                for group in optimizer.param_groups:
                    group["lr"] = current

                onecycle.step()

            del x, y, pred, loss

        if r["lradj"] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r["learning_rate"],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[0]

        train_mse = float(np.mean(losses))

        history.append({
            "Epoch": epoch,
            "TrainMSE": train_mse,
            "LR": lr,
        })

        print(
            f"Fold {name:6s} H={h:3d} F{fold} "
            f"ep={epoch:03d}/{fixed_epochs} "
            f"train={train_mse:.6f} "
            f"lr={lr:.3e}"
        )

    model.eval()

    state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    ckpt = {
        "Dataset": name,
        "Horizon": h,
        "Fold": fold,
        "Prefix": int(prefix),
        "FixedEpochs": int(fixed_epochs),
        "Seed": seed,
        "StateDict": state,
    }

    torch.save(ckpt, path)

    pd.DataFrame(history).to_csv(
        DIRS["history"]
        / f"{name}_H{h}_F{fold}_direct_history.csv",
        index=False,
    )

    return model, ckpt


## 10. Frozen 26-dimensional adaptive-gate features

In [11]:

def score_entropy(s):
    p = torch.softmax(s, dim=1)

    return (
        -(
            p
            * torch.log(
                p.clamp_min(1e-8)
            )
        ).sum(dim=1)
        / math.log(TOP_K)
    )


def feature_cosine(a, b):
    return (
        (a * b).sum(dim=1)
        / (
            torch.sqrt(
                (a * a).sum(dim=1) + 1e-8
            )
            * torch.sqrt(
                (b * b).sum(dim=1) + 1e-8
            )
        )
    )


def gate_features(r, retrieval, direct):
    s = r["score"]
    sim = r["sim"]
    pattern = r["pattern"]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r["cand"].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (cand_std * cand_std).mean(dim=1)
        + 1e-8
    )

    disp_mean = cand_std.mean(dim=1)

    direct_rms = torch.sqrt(
        (direct * direct).mean(dim=1)
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (retrieval * retrieval).mean(dim=1)
        + 1e-8
    )

    disagreement = retrieval - direct

    disagreement_rms = torch.sqrt(
        (disagreement * disagreement).mean(dim=1)
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(dim=1),
            s.std(dim=1, unbiased=False),
            s.max(dim=1).values,
            sorted_s[:, 0] - sorted_s[:, 1],
            s.max(dim=1).values - s.mean(dim=1),
            score_entropy(s),
            sim.mean(dim=1),
            sim.std(dim=1, unbiased=False),
            sim.max(dim=1).values,
            pattern.mean(dim=1),
            pattern.std(dim=1, unbiased=False),
            pattern.max(dim=1).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(direct, retrieval),
        ],
        dim=1,
    )

    out = torch.cat(
        [r["ctx"], scalars],
        dim=1,
    )

    if out.shape[1] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: {out.shape}"
        )

    return out


def abc_terms(direct, retrieval, true):
    e = direct - true
    delta = retrieval - direct

    return torch.stack(
        [
            (e * e).mean(dim=1),
            (e * delta).mean(dim=1),
            (delta * delta).mean(dim=1),
        ],
        dim=1,
    )


## 11. OOF and validation feature collection

In [12]:

@torch.no_grad()
def collect_gate_data(
    data,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    aa,
    h,
):
    C = data["n_channels"]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for i in range(
        0,
        len(aa),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[i:i+FULL_ANCHOR_BATCH]
        A = len(a)

        d3 = direct_residual(
            direct_model,
            z,
            a,
            h,
        )

        pair_anchor = np.repeat(a, C)

        pair_channel = np.tile(
            np.arange(C, dtype=np.int64),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            z,
            pair_anchor,
            pair_channel,
            h,
        )

        retrieval = r["cand"].mean(dim=1)

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(-1, h)

        true = r["true"]

        f = gate_features(
            r,
            retrieval,
            direct,
        )

        a_terms = abc_terms(
            direct,
            retrieval,
            true,
        )

        features.append(
            f.cpu().numpy().astype(np.float32)
        )

        abcs.append(
            a_terms.cpu().numpy().astype(np.float32)
        )

        anchors_out.append(pair_anchor)
        channels_out.append(pair_channel)

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            f,
            a_terms,
        )

    return {
        "feature": np.concatenate(features, axis=0),
        "abc": np.concatenate(abcs, axis=0),
        "anchor": np.concatenate(anchors_out, axis=0),
        "channel": np.concatenate(channels_out, axis=0),
    }


def oof_path(name, h, fold):
    return (
        DIRS["oof"]
        / f"{name}_H{h}_F{fold}_official_direct.npz"
    )


def build_oof_fold(
    data,
    h,
    fold,
    p0,
    p1,
    direct_epochs,
):
    name = data["name"]
    C = data["n_channels"]

    out_path = oof_path(name, h, fold)

    if out_path.exists() and RESUME and not FORCE:
        obj = np.load(out_path)

        print("Loaded OOF:", out_path.name)

        return {
            key: obj[key]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data["train_end"]
    )

    oof_end = int(
        p1
        * data["train_end"]
    )

    z = prefix_norm(
        data["raw"],
        prefix,
    )

    direct_model, _ = train_fold_official_direct(
        name,
        h,
        z,
        prefix,
        direct_epochs,
        fold,
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            h,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        h,
    )

    memory_gpu_obj = memory_gpu(
        retriever,
        memory,
        C,
    )

    aa = eval_anchors(
        prefix,
        oof_end,
        h,
        stride=OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={h} F{fold}: "
        f"prefix={prefix}, end={oof_end}, "
        f"anchors={len(aa)}, pairs={len(aa)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        aa,
        h,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


## 12. Gate training and validation calibration

In [13]:

def fit_feature_scaler(x):
    median = np.median(
        x,
        axis=0,
    ).astype(np.float32)

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (q75 - q25).astype(np.float32)

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(np.float32)

    return median, iqr


def scale_features(x, median, iqr):
    return np.clip(
        (x - median) / iqr,
        -8.0,
        8.0,
    ).astype(np.float32)


def gate_loss(alpha, abc):
    return (
        abc[:, 0]
        + 2.0 * alpha * abc[:, 1]
        + alpha * alpha * abc[:, 2]
    ).mean()


def gate_checkpoint_path(name, h):
    return (
        DIRS["gate"]
        / f"{name}_H{h}_official_direct.pt"
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(len(x))
    losses = []

    for i in range(
        0,
        len(order),
        GATE_BATCH,
    ):
        ids = order[i:i+GATE_BATCH]

        xt = torch.from_numpy(
            x[ids]
        ).to(DEVICE)

        at = torch.from_numpy(
            abc[ids]
        ).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        alpha = model(xt)
        loss = gate_loss(alpha, at)

        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))

        del xt, at, alpha, loss

    return float(np.mean(losses))


@torch.no_grad()
def evaluate_gate(model, x, abc):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(x),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[i:i+GATE_BATCH]
        ).to(DEVICE)

        at = torch.from_numpy(
            abc[i:i+GATE_BATCH]
        ).to(DEVICE)

        alpha = model(xt)

        each = (
            at[:, 0]
            + 2.0 * alpha * at[:, 1]
            + alpha * alpha * at[:, 2]
        )

        total += float(each.sum())
        n += len(alpha)
        alpha_sum += float(alpha.sum())

        del xt, at, alpha, each

    return total / n, alpha_sum / n


def train_crossfit_gate(
    name,
    h,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(name, h)

    if path.exists() and RESUME and not FORCE:
        ckpt = load_torch(path)

        model = CrossFitAdaptiveGate().to(DEVICE)
        model.load_state_dict(ckpt["StateDict"])
        model.eval()

        print("Loaded gate:", path.name)

        return model, ckpt

    median, iqr = fit_feature_scaler(oof_x)

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + h * 3000
        + sum(map(ord, name))
    )

    set_seed(seed)

    model = CrossFitAdaptiveGate().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(seed + 1)

    best = float("inf")
    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch": epoch,
            "OOFTrainMSE": train_mse,
            "ValMSE": val_mse,
            "ValMeanAlpha": mean_alpha,
        })

        if val_mse < best - 1e-10:
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:6s} H={h:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if wait >= GATE_PATIENCE:
            break

    # Reinitialize and fit OOF only for validation-selected epochs.
    set_seed(seed)

    final = CrossFitAdaptiveGate().to(DEVICE)

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(seed + 2)

    for _ in range(best_epoch):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch": best_epoch,
        "BestValMSE": best,
        "FeatureMedian": median,
        "FeatureIQR": iqr,
        "StateDict": {
            k: v.detach().cpu().clone()
            for k, v in final.state_dict().items()
        },
    }

    torch.save(ckpt, path)

    pd.DataFrame(history).to_csv(
        DIRS["history"]
        / f"{name}_H{h}_gate_history.csv",
        index=False,
    )

    return final, ckpt


def mse_scalar(abc, alpha):
    A = abc.astype(np.float64)
    x = float(alpha)

    return float(np.mean(
        A[:, 0]
        + 2.0 * x * A[:, 1]
        + x * x * A[:, 2]
    ))


def choose_scalar(abc):
    rows = []
    best_alpha = None
    best_mse = float("inf")

    for alpha in ALPHA_GRID:
        mse = mse_scalar(abc, alpha)

        rows.append({
            "Alpha": float(alpha),
            "MSE": mse,
        })

        if mse < best_mse - 1e-10:
            best_mse = mse
            best_alpha = float(alpha)

    return best_alpha, pd.DataFrame(rows)


@torch.no_grad()
def gate_alpha(model, ckpt, x):
    sx = scale_features(
        x,
        ckpt["FeatureMedian"],
        ckpt["FeatureIQR"],
    )

    outputs = []

    for i in range(
        0,
        len(sx),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[i:i+GATE_BATCH]
        ).to(DEVICE)

        outputs.append(
            model(t).cpu().numpy()
        )

        del t

    return np.concatenate(outputs).astype(np.float32)


def mse_pair(abc, alpha):
    A = abc.astype(np.float64)
    x = np.asarray(alpha, dtype=np.float64)

    return float(np.mean(
        A[:, 0]
        + 2.0 * x * A[:, 1]
        + x * x * A[:, 2]
    ))


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []
    best_lambda = None
    best_mse = float("inf")

    for lmb in LAMBDA_GRID:
        alpha = (
            (1.0 - float(lmb)) * scalar_alpha
            + float(lmb) * gate_alpha_values
        )

        mse = mse_pair(abc, alpha)

        rows.append({
            "Lambda": float(lmb),
            "MSE": mse,
            "MeanAlpha": float(alpha.mean()),
        })

        if mse < best_mse - 1e-10:
            best_mse = mse
            best_lambda = float(lmb)

    return best_lambda, pd.DataFrame(rows)


## 13. Final all-window test and Oracle diagnostic

In [14]:

def empty_stat():
    return {
        "sse": 0.0,
        "sae": 0.0,
        "n": 0,
    }


def update_stat(stat, pred, true):
    e = pred - true

    stat["sse"] += float((e * e).sum())
    stat["sae"] += float(e.abs().sum())
    stat["n"] += e.numel()


def finish_stat(stat):
    return (
        stat["sse"] / stat["n"],
        stat["sae"] / stat["n"],
    )


def oracle_prediction(direct, retrieval, true):
    e = direct - true
    delta = retrieval - direct

    alpha = torch.clamp(
        -(
            (e * delta).sum(dim=1)
            / (
                (delta * delta).sum(dim=1)
                + 1e-8
            )
        ),
        0.0,
        1.0,
    )

    return (
        direct
        + alpha[:, None] * delta
    )


@torch.no_grad()
def test_evaluate(
    data,
    h,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    C = data["n_channels"]

    aa = eval_anchors(
        data["val_end"],
        data["test_end"],
        h,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {key: empty_stat() for key in keys}
    anchor_mse = {key: [] for key in keys}

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    n_pairs = 0

    for i in range(
        0,
        len(aa),
        FULL_ANCHOR_BATCH,
    ):
        a = aa[i:i+FULL_ANCHOR_BATCH]

        A = len(a)

        d3 = direct_residual(
            direct_model,
            data["z"],
            a,
            h,
        )

        pair_anchor = np.repeat(a, C)

        pair_channel = np.tile(
            np.arange(C, dtype=np.int64),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            data["z"],
            pair_anchor,
            pair_channel,
            h,
        )

        retrieval = r["cand"].mean(dim=1)

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(-1, h)

        true = r["true"]

        scalar = (
            direct
            + scalar_alpha
            * (retrieval - direct)
        )

        features = gate_features(
            r,
            retrieval,
            direct,
        ).cpu().numpy().astype(np.float32)

        scaled = scale_features(
            features,
            gate_ckpt["FeatureMedian"],
            gate_ckpt["FeatureIQR"],
        )

        gate_alpha_values = gate(
            torch.from_numpy(scaled).to(DEVICE)
        )

        shrink_alpha_values = (
            (1.0 - shrink_lambda) * scalar_alpha
            + shrink_lambda * gate_alpha_values
        )

        raw_adaptive = (
            direct
            + gate_alpha_values[:, None]
            * (retrieval - direct)
        )

        shrink_adaptive = (
            direct
            + shrink_alpha_values[:, None]
            * (retrieval - direct)
        )

        oracle = oracle_prediction(
            direct,
            retrieval,
            true,
        )

        predictions = {
            "Direct": direct,
            "Retrieval": retrieval,
            "Scalar": scalar,
            "RawAdaptive": raw_adaptive,
            "ShrinkAdaptive": shrink_adaptive,
            "Oracle": oracle,
        }

        for key, pred in predictions.items():
            update_stat(
                stats[key],
                pred,
                true,
            )

            per_pair = (
                ((pred - true) ** 2)
                .mean(dim=1)
                .reshape(A, C)
                .mean(dim=1)
            )

            anchor_mse[key].extend(
                per_pair.cpu().numpy().tolist()
            )

        raw_alpha_sum += float(gate_alpha_values.sum())
        shrink_alpha_sum += float(shrink_alpha_values.sum())
        n_pairs += len(gate_alpha_values)

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            scalar,
            features,
            scaled,
            gate_alpha_values,
            shrink_alpha_values,
            raw_adaptive,
            shrink_adaptive,
            oracle,
            predictions,
        )

    return {
        "anchors": aa,
        "metrics": {
            key: finish_stat(value)
            for key, value in stats.items()
        },
        "anchor_mse": {
            key: np.asarray(value, dtype=np.float32)
            for key, value in anchor_mse.items()
        },
        "raw_mean_alpha": raw_alpha_sum / n_pairs,
        "shrink_mean_alpha": shrink_alpha_sum / n_pairs,
    }


def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=131313,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(x)
    L = min(block, n)

    rng = np.random.default_rng(seed)

    n_blocks = int(
        np.ceil(n / L)
    )

    max_start = max(
        1,
        n - L + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(n_boot):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [x[s:s+L] for s in starts]
        )[:n]

        boot[b] = sample.mean()

    return {
        "MeanImprovement": float(x.mean()),
        "CI_Low": float(np.quantile(boot, 0.025)),
        "CI_High": float(np.quantile(boot, 0.975)),
    }



## 14. Main experiment

실행 순서는 다음과 같습니다.

$$
H=192 \rightarrow 336 \rightarrow 720
$$

각 horizon에서 ETTm1과 ETTh1을 모두 실행합니다.

중간 결과가 좋거나 나쁘더라도 다음 설정을 수정하지 않습니다.


In [18]:

SUMMARY_PATH = ROOT / "summary.csv"
BOOTSTRAP_PATH = ROOT / "bootstrap.csv"
CALIBRATION_PATH = ROOT / "calibration.csv"
FOLD_PATH = ROOT / "folds.csv"

existing = (
    pd.read_csv(SUMMARY_PATH)
    if RESUME and SUMMARY_PATH.exists()
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict("records")
    if len(existing)
    else []
)

bootstrap_rows = []
calibration_frames = []
fold_rows = []


def already_done(name, h):
    if not len(existing):
        return False

    return bool(
        (
            (existing["Dataset"] == name)
            & (existing["Horizon"] == h)
        ).any()
    )


for h in HORIZONS:
    for name in DATASETS:
        if already_done(name, h):
            print(
                f"SKIP completed: {name} H={h}"
            )
            continue

        t_start = time.time()

        data = DATA[name]
        C = data["n_channels"]

        print("\n" + "#" * 140)
        print(
            f"{name} | OFFICIAL PATCHTST + FROZEN RETRIEVAL | H={h}"
        )
        print("#" * 140)

        # ----------------------------------------------------
        # 1. Train/load official strong direct model.
        # ----------------------------------------------------
        direct_model, direct_ckpt = train_or_load_full_direct(
            name,
            h,
        )

        direct_epochs = int(
            direct_ckpt["BestEpoch"]
        )

        direct_mse_check, direct_mae_check, direct_windows = (
            direct_full_test_metric(
                direct_model,
                data,
                h,
            )
        )

        print(
            f"Strong direct all-window test: "
            f"MSE={direct_mse_check:.6f}, "
            f"MAE={direct_mae_check:.6f}, "
            f"best_epoch={direct_epochs}"
        )

        # ----------------------------------------------------
        # 2. Frozen full retriever.
        # ----------------------------------------------------
        retriever, retriever_ckpt = load_frozen_retriever(
            full_retriever_ckpt_path(
                name,
                h,
            )
        )

        print(
            "Frozen retriever best epoch:",
            retriever_ckpt["BestEpoch"],
        )

        # ----------------------------------------------------
        # 3. OOF data with official fold directs.
        # ----------------------------------------------------
        oof_parts = []

        for fold, (p0, p1) in enumerate(
            FOLDS,
            start=1,
        ):
            part = build_oof_fold(
                data,
                h,
                fold,
                p0,
                p1,
                direct_epochs,
            )

            oof_parts.append(part)

            fold_rows.append({
                "Dataset": name,
                "Horizon": h,
                "Fold": fold,
                "PrefixFrac": p0,
                "OOFEndFrac": p1,
                "Pairs": len(part["feature"]),
                "Anchors": len(
                    np.unique(part["anchor"])
                ),
                "DirectFixedEpochs": direct_epochs,
                "RetrieverCheckpoint": str(
                    fold_retriever_ckpt_path(
                        name,
                        h,
                        fold,
                    )
                ),
            })

        oof_x = np.concatenate(
            [part["feature"] for part in oof_parts],
            axis=0,
        )

        oof_abc = np.concatenate(
            [part["abc"] for part in oof_parts],
            axis=0,
        )

        print("Total OOF pairs:", len(oof_x))

        # ----------------------------------------------------
        # 4. Validation calibration.
        # ----------------------------------------------------
        val_memory = build_memory(
            data["z"],
            C,
            data["train_end"],
            h,
        )

        val_memory_gpu = memory_gpu(
            retriever,
            val_memory,
            C,
        )

        val_aa = eval_anchors(
            data["train_end"],
            data["val_end"],
            h,
            stride=1,
        )

        val_data = collect_gate_data(
            data,
            direct_model,
            retriever,
            val_memory_gpu,
            data["z"],
            val_aa,
            h,
        )

        gate, gate_ckpt = train_crossfit_gate(
            name,
            h,
            oof_x,
            oof_abc,
            val_data["feature"],
            val_data["abc"],
        )

        scalar_alpha, scalar_curve = choose_scalar(
            val_data["abc"]
        )

        raw_val_alpha = gate_alpha(
            gate,
            gate_ckpt,
            val_data["feature"],
        )

        shrink_lambda, lambda_curve = choose_lambda(
            val_data["abc"],
            raw_val_alpha,
            scalar_alpha,
        )

        scalar_curve["Dataset"] = name
        scalar_curve["Horizon"] = h
        scalar_curve["Kind"] = "ScalarAlpha"

        lambda_curve["Dataset"] = name
        lambda_curve["Horizon"] = h
        lambda_curve["Kind"] = "ShrinkLambda"
        lambda_curve["ScalarAlpha"] = scalar_alpha

        calibration_frames.extend(
            [scalar_curve, lambda_curve]
        )

        print(
            f"Validation calibration | "
            f"alpha0={scalar_alpha:.2f} | "
            f"lambda={shrink_lambda:.2f} | "
            f"gateEpoch={gate_ckpt['BestEpoch']}"
        )

        del (
            val_memory,
            val_memory_gpu,
            val_data,
            oof_x,
            oof_abc,
            oof_parts,
            raw_val_alpha,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ----------------------------------------------------
        # 5. Static test memory = train + validation only.
        # ----------------------------------------------------
        test_memory = build_memory(
            data["z"],
            C,
            data["val_end"],
            h,
        )

        test_memory_gpu = memory_gpu(
            retriever,
            test_memory,
            C,
        )

        test = test_evaluate(
            data,
            h,
            direct_model,
            retriever,
            test_memory_gpu,
            gate,
            gate_ckpt,
            scalar_alpha,
            shrink_lambda,
        )

        metrics = test["metrics"]

        direct_mse, direct_mae = metrics["Direct"]
        retrieval_mse, retrieval_mae = metrics["Retrieval"]
        scalar_mse, scalar_mae = metrics["Scalar"]
        raw_mse, raw_mae = metrics["RawAdaptive"]
        shrink_mse, shrink_mae = metrics["ShrinkAdaptive"]
        oracle_mse, oracle_mae = metrics["Oracle"]

        # ----------------------------------------------------
        # 6. Paired moving-block bootstrap.
        # ----------------------------------------------------
        b_direct = moving_block_bootstrap(
            test["anchor_mse"]["Direct"]
            - test["anchor_mse"]["ShrinkAdaptive"],
            seed=131313 + h + sum(map(ord, name)),
        )

        bootstrap_rows.append({
            "Dataset": name,
            "Horizon": h,
            "Comparison": "Direct-ShrinkAdaptive",
            **b_direct,
            "SignificantPositive": b_direct["CI_Low"] > 0.0,
        })

        b_scalar = moving_block_bootstrap(
            test["anchor_mse"]["Scalar"]
            - test["anchor_mse"]["ShrinkAdaptive"],
            seed=232323 + h + sum(map(ord, name)),
        )

        bootstrap_rows.append({
            "Dataset": name,
            "Horizon": h,
            "Comparison": "Scalar-ShrinkAdaptive",
            **b_scalar,
            "SignificantPositive": b_scalar["CI_Low"] > 0.0,
        })

        row = {
            "Dataset": name,
            "Horizon": h,
            "Channels": C,
            "OfficialPatchTST_MSE": direct_mse,
            "OfficialPatchTST_MAE": direct_mae,
            "DirectCheck_MSE": direct_mse_check,
            "DirectParityAbsDiff": abs(
                direct_mse - direct_mse_check
            ),
            "Retrieval_MSE": retrieval_mse,
            "Retrieval_MAE": retrieval_mae,
            "ScalarAlpha": scalar_alpha,
            "Scalar_MSE": scalar_mse,
            "Scalar_MAE": scalar_mae,
            "RawAdaptive_MSE": raw_mse,
            "RawAdaptive_MAE": raw_mae,
            "ShrinkLambda": shrink_lambda,
            "ShrinkAdaptive_MSE": shrink_mse,
            "ShrinkAdaptive_MAE": shrink_mae,
            "Oracle_MSE": oracle_mse,
            "Oracle_MAE": oracle_mae,
            "RawMeanAlpha": test["raw_mean_alpha"],
            "ShrinkMeanAlpha": test["shrink_mean_alpha"],
            "ScalarGainVsDirect_pct": (
                100.0
                * (direct_mse - scalar_mse)
                / direct_mse
            ),
            "ShrinkGainVsDirect_pct": (
                100.0
                * (direct_mse - shrink_mse)
                / direct_mse
            ),
            "ShrinkGainVsScalar_pct": (
                100.0
                * (scalar_mse - shrink_mse)
                / scalar_mse
            ),
            "OracleHeadroomFromDirect_pct": (
                100.0
                * (direct_mse - oracle_mse)
                / direct_mse
            ),
            "OracleHeadroomFromShrink_pct": (
                100.0
                * (shrink_mse - oracle_mse)
                / shrink_mse
            ),
            "OfficialDirectBestEpoch": direct_epochs,
            "FrozenRetrieverBestEpoch": int(
                retriever_ckpt["BestEpoch"]
            ),
            "GateBestEpoch": int(
                gate_ckpt["BestEpoch"]
            ),
            "TestMemoryPerChannel": int(
                test_memory["M"]
            ),
            "TestWindows": len(test["anchors"]),
            "RuntimeMinutes": (
                time.time() - t_start
            ) / 60.0,
        }

        summary_rows.append(row)

        np.savez_compressed(
            DIRS["paired"]
            / f"{name}_H{h}_anchor_mse.npz",
            TestAnchors=test["anchors"],
            Direct=test["anchor_mse"]["Direct"],
            Retrieval=test["anchor_mse"]["Retrieval"],
            Scalar=test["anchor_mse"]["Scalar"],
            RawAdaptive=test["anchor_mse"]["RawAdaptive"],
            ShrinkAdaptive=test["anchor_mse"]["ShrinkAdaptive"],
            Oracle=test["anchor_mse"]["Oracle"],
        )

        pd.DataFrame(summary_rows).to_csv(
            SUMMARY_PATH,
            index=False,
        )

        pd.DataFrame(bootstrap_rows).to_csv(
            BOOTSTRAP_PATH,
            index=False,
        )

        pd.DataFrame(fold_rows).to_csv(
            FOLD_PATH,
            index=False,
        )

        if calibration_frames:
            pd.concat(
                calibration_frames,
                ignore_index=True,
            ).to_csv(
                CALIBRATION_PATH,
                index=False,
            )

        display(pd.DataFrame([row])[
            [
                "Dataset",
                "Horizon",
                "OfficialPatchTST_MSE",
                "Retrieval_MSE",
                "Scalar_MSE",
                "RawAdaptive_MSE",
                "ShrinkAdaptive_MSE",
                "Oracle_MSE",
                "ScalarAlpha",
                "ShrinkLambda",
                "ShrinkGainVsDirect_pct",
                "ShrinkGainVsScalar_pct",
                "ShrinkMeanAlpha",
            ]
        ])

        display(
            pd.DataFrame(bootstrap_rows)[
                (
                    pd.DataFrame(bootstrap_rows)["Dataset"]
                    == name
                )
                & (
                    pd.DataFrame(bootstrap_rows)["Horizon"]
                    == h
                )
            ]
        )

        del (
            direct_model,
            direct_ckpt,
            retriever,
            retriever_ckpt,
            gate,
            gate_ckpt,
            test_memory,
            test_memory_gpu,
            test,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["Horizon", "Dataset"]
    )
    .reset_index(drop=True)
)

display(summary_df)


SKIP completed: ETTm1 H=192
SKIP completed: ETTh1 H=192
SKIP completed: ETTm1 H=336
SKIP completed: ETTh1 H=336
SKIP completed: ETTm1 H=720
SKIP completed: ETTh1 H=720


,Dataset,Horizon,Channels,OfficialPatchTST_MSE,OfficialPatchTST_MAE,DirectCheck_MSE,DirectParityAbsDiff,Retrieval_MSE,Retrieval_MAE,ScalarAlpha,Scalar_MSE,Scalar_MAE,RawAdaptive_MSE,RawAdaptive_MAE,ShrinkLambda,ShrinkAdaptive_MSE,ShrinkAdaptive_MAE,Oracle_MSE,Oracle_MAE,RawMeanAlpha,ShrinkMeanAlpha,ScalarGainVsDirect_pct,ShrinkGainVsDirect_pct,ShrinkGainVsScalar_pct,OracleHeadroomFromDirect_pct,OracleHeadroomFromShrink_pct,OfficialDirectBestEpoch,FrozenRetrieverBestEpoch,GateBestEpoch,TestMemoryPerChannel,TestWindows,RuntimeMinutes
0,ETTh1,192,7,0.413694,0.420978,0.413694,0.0,1.791406,0.866263,0.1,0.429722,0.436829,0.420631,0.429180,1.0,0.420631,0.429180,0.387516,0.409534,0.080383,0.080383,-3.874237,-1.676780,2.115497,6.327950,7.872722,73,5,27,469,2689,3.157023
1,ETTm1,192,7,0.336827,0.372098,0.336827,0.0,1.325316,0.731717,0.0,0.336827,0.372098,0.335848,0.372438,1.0,0.335848,0.372438,0.277740,0.342952,0.056927,0.056927,0.000000,0.290713,0.290713,17.542188,17.301773,22,1,11,1909,11329,14.100202
2,ETTh1,336,7,0.441984,0.441434,0.441984,0.0,1.850639,0.889953,0.1,0.458950,0.457434,0.448468,0.448939,1.0,0.448468,0.448939,0.421000,0.432732,0.079037,0.079037,-3.838419,-1.466894,2.283861,4.747864,6.124911,9,3,29,463,2545,1.622102
3,ETTm1,336,7,0.365620,0.391897,0.365620,0.0,1.364277,0.761060,0.0,0.365620,0.391897,0.365671,0.391823,1.0,0.365671,0.391823,0.325842,0.368720,0.056660,0.056660,0.000000,-0.013909,-0.013909,10.879782,10.892176,17,5,11,1903,11185,11.869051
4,ETTh1,720,7,0.461510,0.473867,0.461510,0.0,1.920814,0.921992,0.1,0.479654,0.489601,0.479586,0.489555,0.0,0.479654,0.489601,0.439043,0.462884,0.099805,0.100000,-3.931421,-3.931421,0.000000,4.868058,8.466621,7,3,1,447,2161,1.535079
5,ETTm1,720,7,0.414840,0.420725,0.414840,0.0,1.424819,0.798014,0.1,0.423251,0.427446,0.420235,0.425324,1.0,0.420235,0.425324,0.380129,0.401218,0.089666,0.089666,-2.027531,-1.300476,0.712606,8.367501,9.543862,13,5,7,1887,10801,10.227085



## 15. Compact horizon-generalization table

가장 중요한 열은 다음 세 개입니다.

- `OfficialPatchTST_MSE`
- `ShrinkAdaptive_MSE`
- `ShrinkGainVsDirect_pct`

그리고 bootstrap에서

$$
\text{CI}_{\mathrm{low}} > 0
$$

이면 Shrinkage Adaptive가 Direct보다 유의하게 좋다고 해석합니다.


In [19]:

if not len(summary_df):
    raise RuntimeError("No completed results.")

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "OfficialPatchTST_MSE",
        "Scalar_MSE",
        "ShrinkAdaptive_MSE",
        "Oracle_MSE",
        "ScalarAlpha",
        "ShrinkLambda",
        "ShrinkMeanAlpha",
        "ShrinkGainVsDirect_pct",
        "OracleHeadroomFromDirect_pct",
    ]
].copy()

compact["Winner"] = np.where(
    compact["ShrinkAdaptive_MSE"]
    < compact["OfficialPatchTST_MSE"],
    "Ours",
    "Direct",
)

display(compact)

if BOOTSTRAP_PATH.exists():
    boot_df = pd.read_csv(
        BOOTSTRAP_PATH
    )

    direct_boot = boot_df[
        boot_df["Comparison"]
        == "Direct-ShrinkAdaptive"
    ][
        [
            "Dataset",
            "Horizon",
            "MeanImprovement",
            "CI_Low",
            "CI_High",
            "SignificantPositive",
        ]
    ]

    display(
        direct_boot.sort_values(
            ["Horizon", "Dataset"]
        )
    )


,Dataset,Horizon,OfficialPatchTST_MSE,Scalar_MSE,ShrinkAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ShrinkGainVsDirect_pct,OracleHeadroomFromDirect_pct,Winner
0,ETTh1,192,0.413694,0.429722,0.420631,0.387516,0.1,1.0,0.080383,-1.676780,6.327950,Direct
1,ETTm1,192,0.336827,0.336827,0.335848,0.277740,0.0,1.0,0.056927,0.290713,17.542188,Ours
2,ETTh1,336,0.441984,0.458950,0.448468,0.421000,0.1,1.0,0.079037,-1.466894,4.747864,Direct
3,ETTm1,336,0.365620,0.365620,0.365671,0.325842,0.0,1.0,0.056660,-0.013909,10.879782,Direct
4,ETTh1,720,0.461510,0.479654,0.479654,0.439043,0.1,0.0,0.100000,-3.931421,4.868058,Direct
5,ETTm1,720,0.414840,0.423251,0.420235,0.380129,0.1,1.0,0.089666,-1.300476,8.367501,Direct


,Dataset,Horizon,MeanImprovement,CI_Low,CI_High,SignificantPositive
2,ETTh1,192,-0.006937,-0.009349,-0.004259,False
0,ETTm1,192,0.000979,-0.000173,0.002151,False
6,ETTh1,336,-0.006483,-0.008331,-0.004493,False
4,ETTm1,336,-0.000051,-0.001015,0.000981,False
10,ETTh1,720,-0.018144,-0.022114,-0.014229,False
8,ETTm1,720,-0.005395,-0.007485,-0.003113,False



# 16. 해석 기준

## 시나리오 A — horizon이 길어질수록 개선이 증가

예를 들어

$$
\Delta_{96} \approx 0,
\qquad
\Delta_{192} > 0,
\qquad
\Delta_{336} > \Delta_{192},
\qquad
\Delta_{720} > \Delta_{336}
$$

와 같은 패턴이 나타나면 매우 중요한 결과입니다.

이는 다음 가설을 지지합니다.

> Short-horizon dynamics are largely captured by the parametric forecaster,
> while external historical memory becomes more complementary as uncertainty accumulates over longer horizons.

---

## 시나리오 B — 일부 long horizon만 개선

이 경우에도 충분히 의미가 있습니다.

다음 세 가지를 함께 봅니다.

1. retrieval-only MSE
2. Oracle headroom
3. adaptive gate의 평균 weight

Oracle headroom이 큰데 실제 방법이 개선되지 않으면,
historical memory의 추가 정보는 존재하지만 trust estimation이 충분하지 않은 것입니다.

---

## 시나리오 C — 모든 horizon에서 실패

ETT에서는 strong PatchTST가 historical retrieval을 충분히 흡수하거나,
현재 retrieval representation이 ETT의 useful analog를 찾지 못한다고 해석합니다.

이 경우 **ETT test 결과를 보고 architecture를 수정하지 않습니다.**

다음 단계는 strong PatchTST를 Weather/Electricity에 적용한 뒤
동일한 frozen augmentation을 검증하는 것입니다.

---

## 중요한 원칙

이번 Experiment 17 결과를 보고 retriever나 gate를 수정하지 않습니다.

이 실험의 목적은 새로운 방법을 찾는 것이 아니라
Experiment 16에서 고정한 방법의 **long-horizon generalization**을 검증하는 것입니다.


## 17. Saved artifacts

In [20]:

print("Experiment root:", ROOT)

for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(ROOT))


Experiment root: /data/dataset/strong_forecaster/official_patchtst_plus_frozen_retrieval_long_horizon
bootstrap.csv
calibration.csv
fold_direct/ETTh1_H192_F1_official_direct.pt
fold_direct/ETTh1_H192_F2_official_direct.pt
fold_direct/ETTh1_H192_F3_official_direct.pt
fold_direct/ETTh1_H336_F1_official_direct.pt
fold_direct/ETTh1_H336_F2_official_direct.pt
fold_direct/ETTh1_H336_F3_official_direct.pt
fold_direct/ETTh1_H720_F1_official_direct.pt
fold_direct/ETTh1_H720_F2_official_direct.pt
fold_direct/ETTh1_H720_F3_official_direct.pt
fold_direct/ETTm1_H192_F1_official_direct.pt
fold_direct/ETTm1_H192_F2_official_direct.pt
fold_direct/ETTm1_H192_F3_official_direct.pt
fold_direct/ETTm1_H336_F1_official_direct.pt
fold_direct/ETTm1_H336_F2_official_direct.pt
fold_direct/ETTm1_H336_F3_official_direct.pt
fold_direct/ETTm1_H720_F1_official_direct.pt
fold_direct/ETTm1_H720_F2_official_direct.pt
fold_direct/ETTm1_H720_F3_official_direct.pt
folds.csv
full_direct/ETTh1_L336_H192_official_recipe_seed